# 02 — Reconstruction libre de w(z) en bins

**Amendement-01 (2026-06-15)** — ce notebook est le livrable
central de la révision méthodologique.

## Objectif
Calculer σ_φ — la grandeur primaire du verdict — sans passer par
CPL. On reconstruit w(z) librement en 4 bins et on teste si les
données imposent physiquement un franchissement w < −1.

## Bins figés (amendement §3, NE PAS MODIFIER)
| Bin | Intervalle z | Contrainte principale |
|---|---|---|
| w₁ | [0.0, 0.3] | BGS |
| w₂ | [0.3, 0.6] | LRG1, LRG2 — **bin pivot** |
| w₃ | [0.6, 1.0] | LRG3+ELG1 |
| w₄ | [1.0, 1.5] | ELG2, QSO |
| w = −1 | z > 1.5 | Figé — énergie noire sous-dominante |

## Sorties de ce notebook
- σ_φ (Pantheon+) et σ_φ (DES-SN5YR)  → entrée dans 03_verdict.ipynb
- P(w < −1) dans le bin pivot [0.3, 0.6] → secondaire, hors verdict
- Cross-check GP (robustesse, hors verdict)

In [1]:
import sys, os
REPO = os.path.expanduser("~/Desktop/souriau-falsification")
os.chdir(REPO)
sys.path.insert(0, REPO)

import numpy as np
import matplotlib.pyplot as plt
from scipy import stats, optimize
from getdist import loadMCSamples
import getdist.plots as gdplt

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11, 'axes.grid': True, 'grid.alpha': 0.3})

# ── Bins figés par l'amendement-01 §3 — NE PAS MODIFIER ──────────
Z_BINS   = [0.0, 0.3, 0.6, 1.0, 1.5]
BIN_NAMES = ['w_bin1', 'w_bin2', 'w_bin3', 'w_bin4']
BIN_LABELS = ['w₁ z=[0.0,0.3]', 'w₂ z=[0.3,0.6]',
               'w₃ z=[0.6,1.0]', 'w₄ z=[1.0,1.5]']
BIN_PIVOT = 'w_bin2'   # bin [0.3,0.6] — LRG — bin le mieux contraint

# ── Chemins des chaînes ───────────────────────────────────────────
CHAINS = {
    'Pantheon+': 'cobaya_runs/tachyon_output/freeform_wz_pantheon/freeform_wz_pantheon',
    'DES-SN5YR': 'cobaya_runs/tachyon_output/freeform_wz_des/freeform_wz_des',
}

print("Setup OK")
print(f"Bins z : {Z_BINS}")
print(f"Bin pivot : {BIN_PIVOT} — z=[0.3, 0.6]")

Setup OK
Bins z : [0.0, 0.3, 0.6, 1.0, 1.5]
Bin pivot : w_bin2 — z=[0.3, 0.6]


In [2]:
available = {}
for label, path in CHAINS.items():
    folder = os.path.dirname(path)
    exists = os.path.exists(folder)
    available[label] = exists
    status = "✓ disponible" if exists else "✗ run pas encore terminé"
    print(f"  {label:<12} : {status}")
    if exists:
        files = [f for f in os.listdir(folder) if f.endswith('.txt')]
        print(f"              → {files}")

if not any(available.values()):
    print("\n⚠️  Aucune chaîne disponible.")
    print("Lance d'abord :")
    print("  cobaya-run cobaya_runs/tachyon_input/freeform_wz_pantheon.yaml")
    print("  cobaya-run cobaya_runs/tachyon_input/freeform_wz_des.yaml")

  Pantheon+    : ✓ disponible
              → []
  DES-SN5YR    : ✓ disponible
              → []


In [ ]:
samples = {}
for label, path in CHAINS.items():
    if available[label]:
        samples[label] = loadMCSamples(path, settings={'ignore_rows': 0.3})
        print(f"\n{label} — paramètres : "
              f"{[p.name for p in samples[label].getParamNames().names]}")

# Graphique : posteriors des 4 bins
if samples:
    fig, axes = plt.subplots(1, 4, figsize=(14, 4))
    colors = {'Pantheon+': '#D85A30', 'DES-SN5YR': '#185FA5'}

    for label, samp in samples.items():
        for i, (bn, bl) in enumerate(zip(BIN_NAMES, BIN_LABELS)):
            if bn in [p.name for p in samp.getParamNames().names]:
                # Densité 1D
                w_vals = samp.samples[:, samp.index(bn)]
                axes[i].hist(w_vals, bins=40, density=True,
                             alpha=0.5, color=colors[label],
                             label=label)
                axes[i].axvline(-1, color='gray', ls='--', lw=1.5)
                axes[i].set_xlabel(bl, fontsize=9)
                axes[i].set_xlim(-2.5, 0.5)
                if i == 0:
                    axes[i].set_ylabel('Densité')

    axes[-1].legend(fontsize=9)
    fig.suptitle('Posteriors des bins w(z) — Reconstruction libre\n'
                 'Ligne grise = w = −1 (non-fantôme)', fontsize=11)
    plt.tight_layout()
    plt.savefig('results/figures/freeform_wz_posteriors.png',
                bbox_inches='tight', dpi=150)
    plt.show()

In [ ]:
def compute_sigma_phi(samp, label=""):
    """
    Calcule σ_φ selon l'amendement-01 §4.

    D = test joint d'inégalité w_i ≥ −1 sur les 4 bins.
    Sous approximation gaussienne-linéaire, D suit une loi
    χ̄² de Chernoff. On retourne l'équivalent gaussien unilatéral.
    """
    bin_names = [bn for bn in BIN_NAMES
                 if bn in [p.name for p in samp.getParamNames().names]]

    if not bin_names:
        print(f"  ⚠️ Bins non trouvés dans {label}")
        return None, None, None

    # Moyennes et covariance postérieure
    w_means = np.array([samp.mean(bn) for bn in bin_names])
    w_cov   = np.array([[samp.cov(pars=[bi, bj])
                          for bj in bin_names]
                          for bi in bin_names])

    print(f"\n{'='*55}")
    print(f"  σ_φ — {label}")
    print(f"{'='*55}")
    for i, (bn, wm) in enumerate(zip(bin_names, w_means)):
        sigma_i = np.sqrt(w_cov[i,i])
        flag = " ← sous −1 !" if wm < -1 else ""
        print(f"  {BIN_LABELS[i]:<20} = {wm:+.4f} ± {sigma_i:.4f}{flag}")

    # Violations : bins où w_mean < −1
    violations = np.maximum(0.0, -(w_means + 1.0))
    n_viol     = int(np.sum(w_means < -1.0))

    # Statistique D (approximation gaussienne-linéaire)
    C_inv = np.linalg.inv(w_cov)
    D     = float(violations @ C_inv @ violations)

    print(f"\n  Bins sous −1  : {n_viol}")
    print(f"  D (stat test) : {D:.4f}")

    if n_viol > 0 and D > 1e-10:
        # Calibration χ̄² de Chernoff
        # Pour K bins violés, loi mélange chi2(0)...chi2(K)
        # Approximation conservatrice : chi2(K)
        p_value   = float(stats.chi2.sf(D, df=max(1, n_viol)))
        sigma_phi = float(stats.norm.isf(p_value / 2.0))
        sigma_phi = max(0.0, sigma_phi)
    else:
        p_value   = 1.0
        sigma_phi = 0.0

    print(f"  p-value       : {p_value:.4f}")
    print(f"  σ_φ           : {sigma_phi:.3f}")
    print()
    print("  Seuils (amendement §5) :")
    print(f"    σ_φ ≥ 3 → DISFAVORISÉE (franchissement physique)")
    print(f"    σ_φ < 2 → pas de franchissement robuste")
    print(f"    2 ≤ σ_φ < 3 → NON CONCLUANT")

    # Bin pivot : P(w_bin2 < -1) — secondaire, hors verdict
    if BIN_PIVOT in bin_names:
        idx = bin_names.index(BIN_PIVOT)
        w2_mean, w2_std = w_means[idx], np.sqrt(w_cov[idx, idx])
        p_w2_negative = float(stats.norm.cdf(-1.0,
                              loc=w2_mean, scale=w2_std))
        print(f"\n  P(w_bin2 < −1) = {p_w2_negative:.4f}"
              f"  [secondaire, hors verdict]")
    else:
        p_w2_negative = None

    return sigma_phi, D, p_w2_negative

# ── Calcul pour chaque catalogue ──────────────────────────────────
sigma_phi_results = {}
for label, samp in samples.items():
    sp, D, p_pivot = compute_sigma_phi(samp, label)
    sigma_phi_results[label] = {
        'sigma_phi': sp, 'D': D, 'p_pivot': p_pivot
    }

In [ ]:
def w_cpl(z, w0, wa):
    return w0 + wa * z / (1 + z)

z_plot  = np.linspace(0, 2, 200)
w0_desi, wa_desi = -0.838, -0.62   # DESI+CMB+Pantheon+, Table V

fig, axes = plt.subplots(1, len(samples), figsize=(7*len(samples), 5))
if len(samples) == 1:
    axes = [axes]

colors = {'Pantheon+': '#D85A30', 'DES-SN5YR': '#185FA5'}

for ax, (label, samp) in zip(axes, samples.items()):
    bin_names = [bn for bn in BIN_NAMES
                 if bn in [p.name for p in samp.getParamNames().names]]
    w_means = np.array([samp.mean(bn) for bn in bin_names])
    w_stds  = np.array([np.sqrt(samp.cov(pars=[bn, bn]))
                        for bn in bin_names])

    # Dessiner les bins en escalier
    z_edges = Z_BINS
    for i, (wm, ws) in enumerate(zip(w_means, w_stds)):
        z_lo, z_hi = z_edges[i], z_edges[i+1]
        ax.fill_between([z_lo, z_hi], [wm-ws, wm-ws], [wm+ws, wm+ws],
                        alpha=0.25, color=colors[label])
        ax.hlines(wm, z_lo, z_hi, color=colors[label], lw=2,
                  label=f'w_bin{i+1} = {wm:.3f}±{ws:.3f}' if i == 0
                  else f'         {wm:.3f}±{ws:.3f}')

    # CPL DESI pour comparaison
    ax.plot(z_plot, w_cpl(z_plot, w0_desi, wa_desi),
            '--', color='gray', lw=1.5, label='CPL DESI+CMB+PP+')

    ax.axhline(-1, color='black', ls=':', lw=1, label='w = −1')
    ax.axhspan(-3, -1, alpha=0.04, color='red')
    ax.text(1.8, -1.8, 'fantôme\n(interdit\ntachyon)',
            ha='center', fontsize=8, color='red', alpha=0.6)

    sp = sigma_phi_results.get(label, {}).get('sigma_phi', None)
    sp_str = f"σ_φ = {sp:.2f}" if sp is not None else "σ_φ = N/A"

    ax.set_xlabel('Redshift z')
    ax.set_ylabel('w(z)')
    ax.set_title(f'Reconstruction libre — {label}\n{sp_str}',
                 fontsize=11)
    ax.set_xlim(0, 2)
    ax.set_ylim(-2.8, 0.3)
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('results/figures/freeform_wz_reconstruction.png',
            bbox_inches='tight', dpi=150)
plt.show()

## Cross-check GP — hors verdict

L'amendement §3 demande un cross-check par processus gaussien (GP)
pour vérifier que σ_φ ne dépend pas du choix "bins".

Ce cross-check est **reporté** — il n'entre pas dans la logique
de verdict. Une discordance marquée bins/GP bascule le verdict
global en "non concluant" (amendement §6.3).

**Implémentation prévue** (à faire après les runs bins) :
- Noyau exponentiel-carré
- Fonction moyenne w = −1
- ℓ_z ~ U[0.2, 0.8], σ_w ~ U[0, 1] (figés par l'amendement)
- Comparer σ_φ(bins) vs σ_φ(GP) : discordance si |Δσ_φ| > 1

In [ ]:
import json

# Construire le résumé σ_φ
sigma_phi_summary = {}
for label, res in sigma_phi_results.items():
    sigma_phi_summary[label] = {
        'sigma_phi'  : res['sigma_phi'],
        'D'          : res['D'],
        'p_pivot'    : res['p_pivot'],
        'z_bins'     : Z_BINS,
        'bin_pivot'  : BIN_PIVOT,
        'bins_below_minus1': {
            bn: float(samples[label].mean(bn))
            for bn in BIN_NAMES
            if bn in [p.name for p in samples[label].getParamNames().names]
            and samples[label].mean(bn) < -1.0
        }
    }

os.makedirs('results', exist_ok=True)
with open('results/sigma_phi.json', 'w') as f:
    json.dump(sigma_phi_summary, f, indent=2)

print("Sauvegardé → results/sigma_phi.json")
print()
print("══════════════════════════════════════════════════")
print("  RÉSUMÉ σ_φ — entrée pour 03_verdict.ipynb")
print("══════════════════════════════════════════════════")
for label, res in sigma_phi_summary.items():
    sp = res['sigma_phi']
    if sp is None:
        interp = "run pas disponible"
    elif sp >= 3:
        interp = "→ FRANCHISSEMENT PHYSIQUE ROBUSTE"
    elif sp >= 2:
        interp = "→ suggestion (non robuste)"
    else:
        interp = "→ pas de franchissement requis"

    print(f"  {label:<12} : σ_φ = "
          f"{'N/A' if sp is None else f'{sp:.3f}':<6}  {interp}")

print()
print("Ces valeurs + ΔDIC + flag P → 03_verdict.ipynb §5-§6")

## Résultats de ce notebook

### σ_φ (grandeur primaire du verdict)
→ Voir cellule 7 pour les valeurs numériques.

### Ce qui va dans 03_verdict.ipynb
- `results/sigma_phi.json` : σ_φ par catalogue SNe
- ΔDIC : à calculer depuis les chaînes tachyon (cellule dédiée dans 03)
- Flag P : φ̇²_max > 0.95 (à lire depuis les chaînes tachyon)

### Ce qui reste à faire
- [ ] Cross-check GP (robustesse, hors verdict)
- [ ] Run sur DES-SN5YR si pas encore disponible
- [ ] Recalibration Monte-Carlo de D si posterior non gaussienne